In [1]:
import io
import itertools
import os

import numpy as np
import sklearn.metrics

import tensorflow as tf
from tensorboard.plugins.hparams import api as hp

import matplotlib.pyplot as plt

# GPU check -- with the "53 CNN (Metal GPU)" kernel this prints a GPU device.
print("TF", tf.__version__, "| devices:",
      [d.device_type for d in tf.config.list_physical_devices()])


TF 2.18.1 | devices: ['CPU', 'GPU']


In [2]:
data_train = np.load(r"data/Dataset/Trousers & Jeans - All - Train.npz")
data_val = np.load(r"data/Dataset/Trousers & Jeans - All - Validation.npz")
data_test = np.load(r"data/Dataset/Trousers & Jeans - All - Test.npz")

In [3]:
images_train = data_train['images']
labels_train = data_train['genders']

images_val = data_val['images']
labels_val = data_val['genders']

images_test = data_test['images']
labels_test = data_test['genders']

In [4]:
images_train = images_train/255.0

images_val = images_val/255.0

images_test = images_test/255.0

In [5]:
EPOCHS = 15
BATCH_SIZE = 64

In [6]:
HP_FILTER_SIZE_1 = hp.HParam('filter_size_1', hp.Discrete([3,5,7]))
HP_FILTER_NUM = hp.HParam('filter_number', hp.Discrete([32,64,96,128]))
HP_FILTER_SIZE_2 = hp.HParam('filter_size_2', hp.Discrete([3,5]))
HP_DENSE_SIZE = hp.HParam('dense_size', hp.Discrete([256,512,1024]))

METRIC_ACCURACY = 'accuracy'

with tf.summary.create_file_writer(r'Logs/Model 1/hparam_tuning/').as_default():
    hp.hparams_config(
        hparams=[HP_FILTER_SIZE_1,HP_FILTER_NUM,HP_FILTER_SIZE_2,HP_DENSE_SIZE],
        metrics=[hp.Metric(METRIC_ACCURACY,display_name='Accuracy')],
    )

2026-08-25 10:18:33.972235: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-08-25 10:18:33.972367: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-08-25 10:18:33.972379: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
I0000 00:00:1787633313.972669 1217699 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1787633313.972821 1217699 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
def train_test_model(hparams,session_num):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(120,90,3)),
        tf.keras.layers.Conv2D(hparams[HP_FILTER_NUM],hparams[HP_FILTER_SIZE_1],activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
        tf.keras.layers.Conv2D(hparams[HP_FILTER_NUM],hparams[HP_FILTER_SIZE_2],activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(hparams[HP_DENSE_SIZE], activation = 'relu'),
        tf.keras.layers.Dense(2),
    ])
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(optimizer ='adam',loss=loss_fn,metrics=['accuracy'])

    log_dir = "Logs/Model 1/fit/"+"run-{}".format(session_num)

    def plot_confusion_matrix(cm, class_names):

        figure = plt.figure(figsize=(12, 12))
        plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title("Confusion matrix")
        plt.colorbar()
        tick_marks = np.arange(len(class_names))
        plt.xticks(tick_marks, class_names, rotation=45)
        plt.yticks(tick_marks, class_names)

        # Normalize the confusion matrix.
        cm = np.around(cm.astype('float') / cm.sum(axis=1)[:, np.newaxis], decimals=2)

        # Use white text if squares are dark; otherwise black.
        threshold = cm.max() / 2.
        for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            color = "white" if cm[i, j] > threshold else "black"
            plt.text(j, i, cm[i, j], horizontalalignment="center", color=color)

        plt.tight_layout()
        plt.ylabel('True label')
        plt.xlabel('Predicted label')

        return figure

    def plot_to_image(figure):
        """Converts the matplotlib plot specified by 'figure' to a PNG image and
        returns it. The supplied figure is closed and inaccessible after this call."""

        # Save the plot to a PNG in memory.
        buf = io.BytesIO()
        plt.savefig(buf, format='png')

        # Closing the figure prevents it from being displayed directly inside the notebook.
        plt.close(figure)

        buf.seek(0)

        # Convert PNG buffer to TF image
        image = tf.image.decode_png(buf.getvalue(), channels=4)

        # Add the batch dimension
        image = tf.expand_dims(image, 0)

        return image

    # Define a file writer variable for logging purposes
    file_writer_cm = tf.summary.create_file_writer(log_dir + '/cm')

    def log_confusion_matrix(epoch, logs):
        # Use the model to predict the values from the validation dataset.
        test_pred_raw = model.predict(images_val)
        test_pred = np.argmax(test_pred_raw, axis=1)

        # Calculate the confusion matrix.
        cm = sklearn.metrics.confusion_matrix(labels_val, test_pred)

        # Log the confusion matrix as an image summary.
        figure = plot_confusion_matrix(cm, class_names=["Male","Female"])
        cm_image = plot_to_image(figure)

        # Log the confusion matrix as an image summary.
        with file_writer_cm.as_default():
            tf.summary.image("Confusion Matrix", cm_image, step=epoch)

    cm_callback = tf.keras.callbacks.LambdaCallback(on_epoch_end=log_confusion_matrix)
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir,histogram_freq=1,profile_batch=0)

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        mode='auto',
        min_delta=0,
        patience=2,
        verbose=0,
        restore_best_weights=True
    )

    model.fit(
        images_train,labels_train,
        epochs = EPOCHS,
        batch_size = BATCH_SIZE , 
        callbacks = [tensorboard_callback,cm_callback,early_stopping],
        validation_data=(images_val,labels_val),
        verbose =2,
    )

    _,accuracy = model.evaluate(images_val,labels_val)
    # Keep only the best model on disk. Saving every trial costs tens of GB on
    # the larger grids; the per-run metrics all live in TensorBoard anyway.
    global BEST_ACCURACY
    save_path = "saved_models/Gender Model 1/Best.keras"
    if accuracy > BEST_ACCURACY:
        BEST_ACCURACY = accuracy
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        model.save(save_path)
        with open("saved_models/Gender Model 1/Best-info.txt", "w") as f:
            f.write("run-{}  val_accuracy={:.4f}\n".format(session_num, accuracy))
            f.write(str({h.name: hparams[h] for h in hparams}) + "\n")
        print("  new best: run-{} val_accuracy={:.4f} -> {}".format(
            session_num, accuracy, save_path))

    return accuracy


In [8]:
#create a function to log the results
def run(log_dir, hparams, session_num):

    with tf.summary.create_file_writer(log_dir).as_default():
        hp.hparams(hparams) #record the values used in this trial
        accuracy = train_test_model(hparams,session_num)
        tf.summary.scalar(METRIC_ACCURACY,accuracy,step=1)
         


In [9]:
BEST_ACCURACY = 0.0
session_num = 1
for filter_size_1 in HP_FILTER_SIZE_1.domain.values:
    for filter_num in HP_FILTER_NUM.domain.values:
        for filter_size_2 in HP_FILTER_SIZE_2.domain.values:
            for dense_size in HP_DENSE_SIZE.domain.values:
                hparams ={
                    HP_FILTER_SIZE_1:filter_size_1,
                    HP_FILTER_NUM:filter_num,
                    HP_FILTER_SIZE_2:filter_size_2,
                    HP_DENSE_SIZE: dense_size
                }
                run_name = "run-%d" %session_num
                print("--- Starting trial: %s" %run_name)
                print({h.name: hparams[h] for h in hparams})
                run("Logs/Model 1/hparam_tuning/"+run_name, hparams,session_num)

                session_num +=1

--- Starting trial: run-1
{'filter_size_1': 3, 'filter_number': 32, 'filter_size_2': 3, 'dense_size': 256}
Epoch 1/15


2026-08-25 10:18:34.932248: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
63/63 - 4s - 70ms/step - accuracy: 0.6501 - loss: 1.2635 - val_accuracy: 0.7420 - val_loss: 0.5139
Epoch 2/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
63/63 - 3s - 46ms/step - accuracy: 0.7945 - loss: 0.4476 - val_accuracy: 0.7860 - val_loss: 0.4560
Epoch 3/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
63/63 - 3s - 42ms/step - accuracy: 0.7717 - loss: 0.5103 - val_accuracy: 0.8220 - val_loss: 0.4823
Epoch 4/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
63/63 - 3s - 40ms/step - accuracy: 0.8134 - loss: 0.4189 - val_accuracy: 0.8060 - val_loss: 0.4495
Epoch 5/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
63/63 - 3s - 40ms/step - accuracy: 0.8506 - loss: 0.3596 - val_accuracy: 0.8180 - val_loss: 0.4133
Epoch 6/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
63/63 - 2s - 39ms/step - accuracy: 0.8556 - loss: 0.3486 - val_accuracy: 0.8340 - val_loss: 0.4277
Epoch 7/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
63/63 - 2s - 39ms/step - accuracy: 0.8656 - loss: 0.3266 - val_acc

KeyboardInterrupt: 

In [ ]:
# NOTE: the %tensorboard magic renders a blank frame inside VS Code -- it builds the
# iframe URL from window.location, which is a vscode-webview:// URL, not localhost.
# Start the server ourselves and expose a real http://localhost:<port> link instead.
import socket
import subprocess
import sys
import time

from IPython.display import HTML, IFrame, display


def show_tensorboard(logdir, port, wait=8):
    def is_up():
        with socket.socket() as s:
            s.settimeout(0.5)
            return s.connect_ex(("127.0.0.1", port)) == 0

    if is_up():
        print(f"TensorBoard already serving on port {port}")
    else:
        subprocess.Popen(
            [sys.executable, "-m", "tensorboard.main",
             "--logdir", logdir, "--port", str(port)],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        for _ in range(wait * 2):
            if is_up():
                break
            time.sleep(0.5)
        print(f"TensorBoard started on port {port} for {logdir!r}")

    url = f"http://localhost:{port}/"
    display(HTML(f'<a href="{url}" target="_blank">Open TensorBoard: {url}</a>'))
    display(IFrame(url, width="100%", height=800))


In [ ]:
show_tensorboard("Logs/Model 1/hparam_tuning", port=6012)

In [ ]:
show_tensorboard("Logs/Model 1/fit", port=6013)